In [ ]:
!pip install monai
import os
import json
import torch
import torch.nn as nn
from torch import amp
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, NormalizeIntensityd,
    RandCropByPosNegLabeld, ConcatItemsd, RandFlipd, RandRotate90d,
    ToTensord, DivisiblePadd, AsDiscrete
)
from monai.data import PersistentDataset, DataLoader
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric, MeanIoU, ConfusionMatrixMetric

# 1. 환경 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_path = "/content/drive/MyDrive/BraTS2023_Dataset" # 실제데이터 있는 로컬 경로로 바꾸기
checkpoint_dir = os.path.join(base_path, "checkpoints")
cache_dir = "/content/persistent_cache"

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(cache_dir, exist_ok=True)

# 2. 전처리 (개선됨: RandCropByPosNegLabeld 적용)
train_transforms = Compose([
    LoadImaged(keys=["t1n", "t1c", "t2w", "t2f", "label"]),
    EnsureChannelFirstd(keys=["t1n", "t1c", "t2w", "t2f", "label"]),
    ConcatItemsd(keys=["t1n", "t1c", "t2w", "t2f"], name="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    # ⭐ 개선 1: 병변(Pos)과 배경(Neg)의 비율을 3:1로 강제 샘플링
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=[96, 96, 96],
        pos=3,
        neg=1,
        num_samples=1, # 배치 사이즈와 연동되도록 1로 설정
        image_key="image",
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    ToTensord(keys=["image", "label"])
])

val_transforms = Compose([
    LoadImaged(keys=["t1n", "t1c", "t2w", "t2f", "label"]),
    EnsureChannelFirstd(keys=["t1n", "t1c", "t2w", "t2f", "label"]),
    ConcatItemsd(keys=["t1n", "t1c", "t2w", "t2f"], name="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    DivisiblePadd(keys=["image", "label"], k=16),
    ToTensord(keys=["image", "label"])
])

# 3. 모델 설정 (개선됨: 채널 용량 확대)
def get_model():
    return UNet(
        spatial_dims=3,
        in_channels=4,
        out_channels=4,
        # ⭐ 개선 2: 모델 복잡도를 높여 복잡한 패턴을 학습하도록 채널 수 조정
        channels=(32, 64, 128, 256, 512),
        strides=(2, 2, 2, 2),
        num_res_units=2
    ).to(device)

def get_loader(split_name, batch_size=4):
    json_path = os.path.join(base_path, f"splits/{split_name}.json")
    if not os.path.exists(json_path): return None
    with open(json_path, "r") as f: data = json.load(f)
    transforms = train_transforms if "train" in split_name else val_transforms
    ds = PersistentDataset(data=data, transform=transforms, cache_dir=cache_dir)
    return DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)

# 4. 학습 함수
post_pred = AsDiscrete(argmax=True, to_onehot=4)
post_label = AsDiscrete(to_onehot=4)

def run_experiment(train_loader, val_loader, model_name, lr, max_epochs=20, patience=7):
    history_path = os.path.join(checkpoint_dir, f"history_{model_name}_lr{lr}.pth")

    model = get_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr) # AdamW가 수렴에 더 안정적일 수 있음

    # ⭐ 개선 3: DiceLoss + CrossEntropyLoss 결합 (픽셀 세부 정확도 향상)
    loss_function = DiceCELoss(to_onehot_y=True, softmax=True, lambda_dice=1.0, lambda_ce=1.0)

    scaler = amp.GradScaler('cuda')

    dice_metric = DiceMetric(include_background=False, reduction="mean")
    iou_metric = MeanIoU(include_background=False, reduction="mean")
    conf_metric = ConfusionMatrixMetric(include_background=False,
                                        metric_name=["precision", "recall"],
                                        reduction="mean")

    best_dice, early_stop_counter = -1, 0
    history = []

    print(f"\n🚀 [실험 시작] {model_name} | LR: {lr}")

    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0
        for batch_data in train_loader:
            inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)

            # 레이블 4가 있다면 3으로 매핑하여 학습 (BraTS 표준 대응)
            labels[labels == 4] = 3

            optimizer.zero_grad()
            with amp.autocast('cuda'):
                outputs = model(inputs)
                loss = loss_function(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()

        # 검증 섹션
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                v_in, v_lab = val_data["image"].to(device), val_data["label"].to(device)
                v_lab[v_lab == 4] = 3

                with amp.autocast('cuda'):
                    # 메모리 관리를 위해 검증 시 sliding_window_inference 고려 가능 (현재는 전체 입력)
                    v_out = model(v_in)

                val_outputs = [post_pred(i) for i in v_out]
                val_labels = [post_label(i) for i in v_lab]

                dice_metric(y_pred=val_outputs, y=val_labels)
                iou_metric(y_pred=val_outputs, y=val_labels)
                conf_metric(y_pred=val_outputs, y=val_labels)

            dice_res = dice_metric.aggregate().item()
            iou_res = iou_metric.aggregate().item()
            conf_res = conf_metric.aggregate()
            prec_res, recall_res = conf_res[0].item(), conf_res[1].item()

            dice_metric.reset(); iou_metric.reset(); conf_metric.reset()

        avg_loss = epoch_loss / len(train_loader)
        epoch_result = {
            "epoch": epoch + 1, "loss": avg_loss, "dice_f1": dice_res,
            "iou": iou_res, "precision": prec_res, "recall": recall_res
        }
        history.append(epoch_result)
        torch.save(history, history_path)

        print(f"✅ E{epoch+1:02d} | Loss: {avg_loss:.4f} | Dice: {dice_res:.4f} | IoU: {iou_res:.4f} | Prec: {prec_res:.4f} | Rec: {recall_res:.4f}")

        # Early Stopping
        if dice_res > best_dice:
            best_dice = dice_res
            early_stop_counter = 0
            torch.save(model.state_dict(), os.path.join(checkpoint_dir, f"best_{model_name}_lr{lr}.pth"))
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                print(f"🛑 조기 종료 (Patience {patience})")
                break

# 5. 실행 (이미지의 실험 체크포인트 목록 반영)
# 실험할 데이터 규모와 학습률(LR) 리스트를 튜플로 정의합니다.
experiments = [
    ("train_50", [0.0001]),           # 50케이스 실험
    ("train_100", [0.0001]),          # 100케이스 실험
    ("train_150", [0.0001, 1e-05])    # 150케이스 실험 (두 가지 학습률)
]

# 공통 테스트 로더 설정
test_loader = get_loader("test_common", batch_size=1)

for split, lrs in experiments:
    # 각 데이터 규모에 맞는 로더 생성
    loader = get_loader(split, batch_size=4)
    
    if loader:
        for lr in lrs:
            # 이미지에 있는 파일명(MEN_best_...) 형식을 맞추기 위해 
            # model_name 자리에 "MEN_" 접두사를 붙여서 실행합니다.
            experiment_label = f"MEN_{split}" 
            run_experiment(
                loader, 
                test_loader, 
                experiment_label, 
                lr=lr, 
                max_epochs=20, 
                patience=7
            )

print("\n🏁 이미지에 기록된 모든 대조 실험이 완료되었습니다!")